# **보험 약관 기반 RAG 파이프라인**

---

#### **Task 0. 환경 설정 및 라이브러리 import**

In [1]:
!pip install -U langchain-openai langchain-chroma langchain-community langchain-text-splitters langchain-core pypdf pymupdf pdfplumber tiktoken python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 831.9 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 

In [2]:
import os
import re
import glob
import shutil
from pathlib import Path
from pprint import pprint

import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go

from dotenv import load_dotenv

# LangChain / OpenAI
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, PDFPlumberLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# LCEL
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 토큰 수 확인용
import tiktoken

<br>

#### **Task 1. API 키 및 기본 환경 확인**

In [3]:
import os
from google.colab import userdata

# 코랩 보안 비밀(열쇠 아이콘)에서 'OPENAI_API_KEY'라는 이름으로 저장된 값을 가져옴.
try:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    print("OPENAI_API_KEY 로드 완료:", OPENAI_API_KEY[:8] + "...")
except userdata.SecretNotFoundError:
    print("에러: 코랩 보안 비밀에 'OPENAI_API_KEY'가 없습니다.")
    print("왼쪽 열쇠 아이콘을 눌러 키 이름을 확인하고 '노트북 액세스'를 켰는지 확인하세요.")
except userdata.NotebookAccessError:
    print("에러: '노트북 액세스' 스위치가 꺼져 있습니다.")

OPENAI_API_KEY 로드 완료: sk-proj-...


<br>

#### **Task 2. 프로젝트 설정값(CONFIG)**

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
CONFIG = {
    # 문서 로드
    "pdf_path": "/content/drive/MyDrive/DS_8th/09_LLM 활용/data/KB손해보험_가입상품_약관(금경만).pdf",
    "loader_type": "default",     # "default" | "fast" | "detail"

    # 청킹
    "chunk_size": 800,
    "chunk_overlap": 100,
    "separators": ["\n\n", "\n", "제", ".", " "],

    # 임베딩
    "embedding_model": "text-embedding-3-large",  # small -> large

    # 벡터DB
    "vectorstore_dir": "./chroma_insurance_v2",
    "reset_vectorstore": False,   # True면 기존 벡터DB 삭제 후 재구축

    # Retriever
    "retriever_k": 5,

    # LLM
    "llm_model": "gpt-4o-mini",
    "llm_temperature": 0,

    # 토큰 점검용 모델
    "token_count_model": "gpt-4o-mini",
}

print("현재 CONFIG")
pprint(CONFIG)

현재 CONFIG
{'chunk_overlap': 100,
 'chunk_size': 800,
 'embedding_model': 'text-embedding-3-large',
 'llm_model': 'gpt-4o-mini',
 'llm_temperature': 0,
 'loader_type': 'default',
 'pdf_path': '/content/drive/MyDrive/DS_8th/09_LLM '
             '활용/data/KB손해보험_가입상품_약관(금경만).pdf',
 'reset_vectorstore': False,
 'retriever_k': 5,
 'separators': ['\n\n', '\n', '제', '.', ' '],
 'token_count_model': 'gpt-4o-mini',
 'vectorstore_dir': './chroma_insurance_v2'}


<br><br>

#### **Task 3. PDF 로더 선택 및 문서 점검 함수**

In [6]:
LOADER_MAP = {
    "default": PyPDFLoader,
    "fast": PyMuPDFLoader,
    "detail": PDFPlumberLoader,
}

def load_pdf_pages(config=CONFIG):
    """
    PDF를 페이지 단위 Document 리스트로 로드
    """
    pdf_path = Path(config["pdf_path"])
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF 파일이 없습니다: {pdf_path.resolve()}")

    loader_cls = LOADER_MAP.get(config["loader_type"], PyPDFLoader)
    loader = loader_cls(str(pdf_path))
    pages = loader.load()

    print(f"[PDF 로드 완료] loader={loader_cls.__name__}, pages={len(pages)}")
    return pages


def inspect_pages(pages, config=CONFIG, preview_chars=300):
    """
    문서 규모 및 토큰 수 점검
    """
    all_text = "\n\n".join([p.page_content for p in pages])
    total_chars = len(all_text)

    encoding = tiktoken.encoding_for_model(config["token_count_model"])
    total_tokens = len(encoding.encode(all_text))

    print("[문서 점검]")
    print(f"- 총 페이지 수: {len(pages)}")
    print(f"- 총 문자 수: {total_chars:,}")
    print(f"- 총 토큰 수({config['token_count_model']} 기준): {total_tokens:,}")
    print("\n[첫 페이지 미리보기]")
    print(pages[0].page_content[:preview_chars])

<br>

### **개선 1. 불필요 페이지 제거**

In [7]:
EXCLUDE_KEYWORDS = [
    "목차",
    "고객콜센터",
    "전국고객지원센터",
    "보험금지급절차안내",
    "보험금 청구서",
    "청구서류",
    "신분증",
    "지급되지 아니할 수 있습니다",
    "예정이율",
    "환급금",
    "개인신용정보 제공이용",
]

def is_noise_page(text: str) -> bool:
    text_no_space = text.replace(" ", "").replace("\n", "")
    hit_count = sum(kw.replace(" ", "") in text_no_space for kw in EXCLUDE_KEYWORDS)

    # 너무 짧고 안내성 페이지인 경우 제거
    if hit_count >= 2:
        return True

    return False


def filter_noise_pages(pages):
    kept = []
    removed = []

    for page in pages:
        if is_noise_page(page.page_content):
            removed.append(page)
        else:
            kept.append(page)

    print(f"[노이즈 페이지 제거] kept={len(kept)}, removed={len(removed)}")
    return kept, removed

<br>

#### **Task 4. 보험 약관용 메타데이터 보강 함수**

In [8]:
#   조항/유형/키워드 힌트를 추가

def extract_clause_title(text: str) -> str:
    """
    '제3조', '제 3 조' 같은 조항 제목을 최대한 추출
    """
    if not text:
        return "UNKNOWN"

    pattern = r"(제\s*\d+\s*조(?:의\s*\d+)?)"
    m = re.search(pattern, text)
    if m:
        return m.group(1).replace(" ", "")
    return "UNKNOWN"


def infer_insurance_topic(text: str) -> str:
    """
    간단한 키워드 기반 토픽 태깅
    나중에 더 고도화 가능
    """
    text = text.replace(" ", "")

    if "입원" in text:
        return "입원"
    elif "통원" in text:
        return "통원"
    elif "약제" in text or "약국" in text:
        return "약제비"
    elif "비급여" in text:
        return "비급여"
    elif "면책" in text or "보상하지아니" in text or "보상하여드리지아니" in text:
        return "면책/제외"
    elif "수술" in text:
        return "수술"
    else:
        return "기타"


def enrich_page_metadata(pages):
    """
    페이지 단위 metadata 보강
    """
    enriched_pages = []

    for page in pages:
        text = page.page_content
        page.metadata["clause_hint"] = extract_clause_title(text)
        page.metadata["topic_hint"] = infer_insurance_topic(text)
        enriched_pages.append(page)

    print(f"[페이지 메타데이터 보강 완료] {len(enriched_pages)} pages")
    return enriched_pages

<br>

#### **Task 5-1. 청킹 함수**

In [9]:
# 보험 약관은 조항 구조가 중요하므로 separators를 세심하게 설정

def split_documents(pages, config=CONFIG):
    """CONFIG 기반으로 청킹"""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config["chunk_size"],
        chunk_overlap=config["chunk_overlap"],
        separators=config["separators"],
    )
    docs = splitter.split_documents(pages)

    print(f"[청킹 완료] {len(docs)} chunks")
    return docs

<br>

#### **5-2. 청크 메타데이터 보강**

In [10]:
# 검색 품질 향상을 위해 chunk별로 조항/토픽 metadata 추가

def enrich_chunk_metadata(docs):
    enriched_docs = []

    for idx, doc in enumerate(docs):
        text = doc.page_content

        doc.metadata["chunk_id"] = idx
        doc.metadata["clause_hint"] = extract_clause_title(text)
        doc.metadata["topic_hint"] = infer_insurance_topic(text)

        # 검색/출력 확인용 길이
        doc.metadata["char_len"] = len(text)

        enriched_docs.append(doc)

    print(f"[청크 메타데이터 보강 완료] {len(enriched_docs)} chunks")
    return enriched_docs


def preview_chunks(docs, n=3, preview_chars=200):
    print(f"[청크 미리보기] 상위 {n}개")
    for i, doc in enumerate(docs[:n], 1):
        print("=" * 70)
        print(f"chunk_id   : {doc.metadata.get('chunk_id')}")
        print(f"page       : {doc.metadata.get('page')}")
        print(f"clause_hint: {doc.metadata.get('clause_hint')}")
        print(f"topic_hint : {doc.metadata.get('topic_hint')}")
        print(f"text       : {doc.page_content[:preview_chars]}")

<br>

#### **Task 6-1. 임베딩 및 벡터DB 구축/로드**

In [11]:
# 실험 편의성을 위해 reset 옵션 반영

def get_embedding(config=CONFIG):
    return OpenAIEmbeddings(model=config["embedding_model"])


def reset_vectorstore_dir(path: str):
    if os.path.exists(path):
        shutil.rmtree(path)
        print(f"[벡터DB 디렉토리 삭제] {path}")


def build_vectorstore(docs, config=CONFIG):
    if config["reset_vectorstore"]:
        reset_vectorstore_dir(config["vectorstore_dir"])

    embedding = get_embedding(config)
    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=embedding,
        persist_directory=config["vectorstore_dir"],
    )
    print(f"[벡터DB 구축 완료] dir={config['vectorstore_dir']}, chunks={len(docs)}")
    return vectorstore


def load_vectorstore(config=CONFIG):
    embedding = get_embedding(config)
    vectorstore = Chroma(
        persist_directory=config["vectorstore_dir"],
        embedding_function=embedding,
    )
    print(f"[벡터DB 로드 완료] dir={config['vectorstore_dir']}")
    return vectorstore

<br>

#### **Task 6-2. 벡터DB 점검 및 2D/3D 시각화**

- **Task 11**에서 `initialize(rebuild=True)` 함수 실행해, `build_vectorstore(docs, config)` 실행 후, 실행 가능합니다.

In [21]:
collection = _vectorstore._collection

count = collection.count()
sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)

print(f"저장된 벡터 수: {count}")
print(f"벡터 차원 수: {dimensions}")

저장된 벡터 수: 438
벡터 차원 수: 3072


In [22]:
# 벡터 / 문서 / 메타데이터 추출

result = collection.get(include=["embeddings", "documents", "metadatas"])

vectors = np.array(result["embeddings"])
documents = result["documents"]
metadatas = result["metadatas"]

print("vectors shape:", vectors.shape)
print("documents 수:", len(documents))
print("metadatas 수:", len(metadatas))

vectors shape: (438, 3072)
documents 수: 438
metadatas 수: 438


In [23]:
# 2D 시각화

tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(vectors)-1))
reduced_vectors_2d = tsne.fit_transform(vectors)

fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors_2d[:, 0],
    y=reduced_vectors_2d[:, 1],
    mode="markers",
    text=[
        f"page: {m.get('page', '?')}<br>"
        f"clause: {m.get('clause_hint', 'UNKNOWN')}<br>"
        f"topic: {m.get('topic_hint', 'UNKNOWN')}<br>"
        f"text: {d[:100]}..."
        for m, d in zip(metadatas, documents)
    ],
    hoverinfo="text"
)])

fig.update_layout(
    title="2D Chroma Vector Store Visualization",
    width=900,
    height=650,
    margin=dict(r=20, b=10, l=10, t=40),
    xaxis_title="t-SNE dim 1",
    yaxis_title="t-SNE dim 2",
)

fig.show()

In [24]:
# 3D 시각화

tsne = TSNE(n_components=3, random_state=42, perplexity=min(30, len(vectors)-1))
reduced_vectors_3d = tsne.fit_transform(vectors)

fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors_3d[:, 0],
    y=reduced_vectors_3d[:, 1],
    z=reduced_vectors_3d[:, 2],
    mode="markers",
    text=[
        f"page: {m.get('page', '?')}<br>"
        f"clause: {m.get('clause_hint', 'UNKNOWN')}<br>"
        f"topic: {m.get('topic_hint', 'UNKNOWN')}<br>"
        f"text: {d[:100]}..."
        for m, d in zip(metadatas, documents)
    ],
    hoverinfo="text"
)])

fig.update_layout(
    title="3D Chroma Vector Store Visualization",
    width=1000,
    height=750,
    margin=dict(r=10, b=10, l=10, t=40),
    scene=dict(
        xaxis_title="t-SNE dim 1",
        yaxis_title="t-SNE dim 2",
        zaxis_title="t-SNE dim 3",
    ),
)

fig.show()

<br>

#### **Task 7. Retriever 생성 및 검색 품질 점검 함수**

In [14]:
# "답변" 보기 전에 "검색이 맞는지" 먼저 점검 가능하게

def get_retriever(vectorstore, config=CONFIG):
    retriever = vectorstore.as_retriever(
        search_kwargs={"k": config["retriever_k"]}
    )
    print(f"[Retriever 생성] k={config['retriever_k']}")
    return retriever


def inspect_retrieval(retriever, query: str, preview_chars=180):
    """
    답변 생성 전, 검색된 문서가 적절한지 직접 점검
    """
    docs = retriever.invoke(query)

    print("=" * 80)
    print(f"[검색 점검] query: {query}")
    print("=" * 80)

    for i, doc in enumerate(docs, 1):
        print(f"[{i}] page={doc.metadata.get('page')} | clause={doc.metadata.get('clause_hint')} | topic={doc.metadata.get('topic_hint')}")
        print(doc.page_content[:preview_chars])
        print("-" * 80)

    return docs

<br>

#### **Task 8. 프롬프트 및 RAG 체인 구성**

In [15]:
SYSTEM_PROMPT = """
    당신은 보험 약관 해석 보조 AI입니다.
    반드시 검색된 약관 내용에 근거해서만 답변하세요.

    [역할]
    - 사용자의 질문에 대해 검색된 약관 조항을 근거로 답변합니다.
    - 보험금 계산이 필요한 경우, 약관에 명시된 내용까지만 설명합니다.
    - 약관에 없는 내용을 추정해서 단정하지 마세요.

    [반드시 지킬 규칙]
    1. 검색된 조항에 근거가 있으면 그 내용만 요약해서 답하세요.
    2. 검색된 조항에 근거가 불충분하면 "제공된 약관에서 명확히 확인되지 않습니다"라고 답하세요.
    3. 조항 번호(예: 제3조)를 최대한 명시하세요.
    4. 계산이 필요한 경우에도, 근거 조항 없이 임의 계산하지 마세요.
    5. 마지막에 반드시 아래 문장을 덧붙이세요:
    "※ 본 답변은 참고용이며, 실제 보험금 지급 여부 및 금액은 보험사 심사 결과에 따라 달라질 수 있습니다."

    [출력 형식]
    1. 보상 가능 여부
    2. 근거 조항
    3. 설명
    4. 주의사항

    [검색된 약관 내용]
    {context}
"""

In [16]:
def format_docs(docs):
    parts = []
    for i, doc in enumerate(docs, 1):
        page = doc.metadata.get("page", "?")
        clause = doc.metadata.get("clause_hint", "UNKNOWN")
        topic = doc.metadata.get("topic_hint", "기타")

        parts.append(
            f"[검색결과 {i}] page={page}, clause={clause}, topic={topic}\n{doc.page_content}"
        )

    return "\n\n---\n\n".join(parts)


def build_chain(retriever, config=CONFIG):
    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("human", "{question}"),
    ])

    llm = ChatOpenAI(
        model=config["llm_model"],
        temperature=config["llm_temperature"],
    )

    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    print(f"[RAG 체인 구성 완료] model={config['llm_model']}")
    return chain

<br>

#### **Task 9. 전체 초기화 및 실행 인터페이스**

In [17]:
_chain = None
_retriever = None
_vectorstore = None

def initialize(config=CONFIG, rebuild=False):
    """
    전체 파이프라인 초기화
    rebuild=True이면 PDF부터 다시 처리
    """
    global _chain, _retriever, _vectorstore

    if rebuild or (not os.path.exists(config["vectorstore_dir"])):
        pages = load_pdf_pages(config)
        inspect_pages(pages, config)

        pages, removed_pages = filter_noise_pages(pages)
        pages = enrich_page_metadata(pages)

        docs = split_documents(pages, config)
        docs = enrich_chunk_metadata(docs)
        preview_chunks(docs, n=3)

        _vectorstore = build_vectorstore(docs, config)
    else:
        _vectorstore = load_vectorstore(config)

    _retriever = get_retriever(_vectorstore, config)
    _chain = build_chain(_retriever, config)

    print("[초기화 완료]")


def ask(question: str) -> str:
    if _chain is None:
        raise RuntimeError("initialize()를 먼저 호출하세요.")
    return _chain.invoke(question)


def ask_with_sources(question: str) -> dict:
    if _chain is None or _retriever is None:
        raise RuntimeError("initialize()를 먼저 호출하세요.")

    sources = _retriever.invoke(question)
    answer = _chain.invoke(question)

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
    }


def ask_stream(question: str):
    if _chain is None:
        raise RuntimeError("initialize()를 먼저 호출하세요.")

    for chunk in _chain.stream(question):
        print(chunk, end="", flush=True)

<br>

#### **Task 10-1. 검색 결과와 응답을 함께 보기 위한 출력 헬퍼**

In [18]:
# 분석/검증 품질 보완용

def print_result(result: dict, preview_chars=150):
    print("=" * 100)
    print(f"Q. {result['question']}")
    print("=" * 100)
    print("\n[A]")
    print(result["answer"])

    print("\n[참조 문서]")
    for i, doc in enumerate(result["sources"], 1):
        print(
            f"[{i}] page={doc.metadata.get('page')} | "
            f"clause={doc.metadata.get('clause_hint')} | "
            f"topic={doc.metadata.get('topic_hint')}"
        )
        print(doc.page_content[:preview_chars])
        print("-" * 80)

<br>

#### **Task 10-2. 실험용 함수**

In [19]:
# chunk_size, retriever_k, loader_type 등을 바꿔가며 비교

def run_experiment(overrides: dict, questions: list[str]):
    test_config = {**CONFIG, **overrides}

    print("\n" + "=" * 80)
    print("[실험 시작]")
    for k, v in overrides.items():
        print(f"- {k}: {CONFIG.get(k)} -> {v}")
    print("=" * 80)

    initialize(config=test_config, rebuild=True)

    for q in questions:
        result = ask_with_sources(q)
        print_result(result)

    print("[실험 종료]")

<br>

#### **Task 11. 기본 실행**

In [20]:
initialize(rebuild=True)

[PDF 로드 완료] loader=PyPDFLoader, pages=178
[문서 점검]
- 총 페이지 수: 178
- 총 문자 수: 333,378
- 총 토큰 수(gpt-4o-mini 기준): 218,592

[첫 페이지 미리보기]
무배당 
L I G 행 복 한 인 생 보 험(L1 2 . 0 4 ) 
보험안내서 및 약관 
보종 21373 
장기상품팀 2012년 4월 제작 
무배당 LIG행복한인생보험(L12.04)
서울시 강남구 역삼동 649-11 LIG 타워 
고객콜센터 1544-0114 
장기 12-21373-1 
무배당 
L I G 행 복 한 인 생 보 험(L1 2 . 0 4 ) 
보험안내서 및 약관 
보종 21373 
장기상품팀 2012년 4월 제작 
무배당 LIG행복한인생보험(L12.04)
서울시 강남구 역삼동 649-11 LIG 타워 
고객콜센터 15
[노이즈 페이지 제거] kept=149, removed=29
[페이지 메타데이터 보강 완료] 149 pages
[청킹 완료] 438 chunks
[청크 메타데이터 보강 완료] 438 chunks
[청크 미리보기] 상위 3개
chunk_id   : 0
page       : 0
clause_hint: UNKNOWN
topic_hint : 기타
text       : 무배당 
L I G 행 복 한 인 생 보 험(L1 2 . 0 4 ) 
보험안내서 및 약관 
보종 21373 
장기상품팀 2012년 4월 제작 
무배당 LIG행복한인생보험(L12.04)
서울시 강남구 역삼동 649-11 LIG 타워 
고객콜센터 1544-0114 
장기 12-21373-1 
무배당 
L I G 행 복 한 인 생 보 험(L1 2 . 0 4 ) 
chunk_id   : 1
page       : 2
clause_hint: UNKNOWN
topic_hint : 수술
text       : 12. 급성심근경색증진단비 특별약관【비갱신 및 갱신계약】 ··················73
13. 말기간경화진단비 특별약관 ···········

<br>

#### **Task 12. 검색만 먼저 점검**

In [25]:
# 답변 보기 전에 retrieval 품질부터 확인

inspect_retrieval(_retriever, "입원 5일 했을 때 보험금 얼마 받을 수 있어?")
inspect_retrieval(_retriever, "비급여 치료비도 보상이 돼?")
inspect_retrieval(_retriever, "통원 치료 시 공제금액은 얼마야?")

[검색 점검] query: 입원 5일 했을 때 보험금 얼마 받을 수 있어?
[1] page=133 | clause=제15조 | topic=입원
를 계산합니다.
ꊷ 제4항에도 불구하고 청약일 이전에 진단된 질병이라 하더라도 청약일 이후 
5년이 지나는 동안(계약이 자동갱신되어 5년을 지나는 경우를 포함합니다) 그 
질병으로 인하여 추가적인 진단(단순건강검진 제외) 또는 치료사실이 없을 경
우, 청약일로부터 5년이 지난 이후에는 이 약관에 따라 보상하여 드립니다.

--------------------------------------------------------------------------------
[2] page=54 | clause=UNKNOWN | topic=입원
부터 180일이 경과하도록 퇴원없이 계속 입원중인 경우에는 입원일당이 지급
된 최종입원일의 그 다음날을 퇴원일로 봅니다.
최초입원일 입원일당이 지급된 
최종입원일 보장재개
퇴원없이
계속 입원 …
보장
(180일)
보장제외
(180일)
보장
(180일) …
ꊴ 제1항의 경우 피보험자(보험대상자)가 질병에 대한 보장개시일(책
--------------------------------------------------------------------------------
[3] page=131 | clause=제3조 | topic=입원
를 계산합니다.
【의료법 제3조(의료기관)】
이 법에서 의료기관이라 함은 의료인이 공중 또는 특수 다수인을 위하여 의
료・조산의 업을 행하는 곳을 말합니다. 의료기관의 종별은 종합병원・병
원・치과병원・한방병원・요양병원・의원・치과의원・한의원 및 조산원으
로 나누어집니다.
(3) 질병입원형
ꊱ 회사는 피보험자(보험대상자)가 
--------------------------------------------------------------------------------
[4] page=101 | clause=제3조 | topic=입원
LIG손해보험 101
받은 때에는 최

[Document(id='89cd6b84-cdcc-47bd-a041-d31c32e3cf3a', metadata={'total_pages': 178, 'producer': 'PDFium', 'source': '/content/drive/MyDrive/DS_8th/09_LLM 활용/data/KB손해보험_가입상품_약관(금경만).pdf', 'creator': 'PDFium', 'char_len': 628, 'creationdate': '', 'clause_hint': 'UNKNOWN', 'topic_hint': '통원', 'page_label': '136', 'chunk_id': 354, 'page': 135}, page_content='여 30만원을 최고한도로 계약자가 정하는 금액으로 합니다)을 한도로 보상하\n여 드립니다.\nꊵ 피보험자(보험대상자)가 통원하여 치료를 받던 중 보험기간이 만료되더라\n도 그 계속 중인 통원 치료에 대하여는 보험기간 만료일로부터 180일 이내에 \n외래는 방문 90회, 처방조제비는 처방전 90건을 한도로 보상하여 드립니다.\n<보장기간 예시>\n보장대상기간\n(1년)\n보장대상기간\n(1년)\n보장대상기간\n(1년)\n추가보상\n(180일)\n↑\n계약일\n(2010.1.1)\n↑\n계약해당일\n(2011.1.1)\n↑\n계약해당일\n(2012.1.1)\n↑\n보험기간\n종료일\n(2012.12.31)\n↑\n보상종료\n(2013.6.30)\nꊶ 하나의 상해 또는 하나의 질병으로 인해 하루에 같은 치료를 목적으로 의\n료기관에 2회 이상 통원치료시(하나의 상해 또는 하나의 질병으로 약국을 통\n한 2회 이상의 처방조제를 포함합니다) 1회의 외래 및 1건의 처방으로 간주하\n여 제1항 및 제5항을 적용합니다. 이 때 외래에 대한 공제금액은 2회 이상의 \n중복방문 의료기관 중 가장 높은 공제금액을 적용합니다.\nꊷ 피보험자(보험대상자)가 병원 또는 약국의 직원복리후생제도에 의하여 납\n부할 의료비를 감면받은 경우에는 그 감면 전

<br>

#### **Task 13. 질의응답 테스트**

In [26]:
test_questions = [
    "입원 5일 했을 때 보험금 얼마 받을 수 있어?",
    "비급여 치료비도 보상이 돼?",
    "통원 치료 시 공제금액은 얼마야?",
]

for q in test_questions:
    result = ask_with_sources(q)
    print_result(result)

Q. 입원 5일 했을 때 보험금 얼마 받을 수 있어?

[A]
제공된 약관에서 명확히 확인되지 않습니다. 

※ 본 답변은 참고용이며, 실제 보험금 지급 여부 및 금액은 보험사 심사 결과에 따라 달라질 수 있습니다.

[참조 문서]
[1] page=133 | clause=제15조 | topic=입원
를 계산합니다.
ꊷ 제4항에도 불구하고 청약일 이전에 진단된 질병이라 하더라도 청약일 이후 
5년이 지나는 동안(계약이 자동갱신되어 5년을 지나는 경우를 포함합니다) 그 
질병으로 인하여 추가적인 진단(단순건강검진 제외) 또는 치료사실이 없을 경
우, 청약일로부터 5년
--------------------------------------------------------------------------------
[2] page=54 | clause=UNKNOWN | topic=입원
부터 180일이 경과하도록 퇴원없이 계속 입원중인 경우에는 입원일당이 지급
된 최종입원일의 그 다음날을 퇴원일로 봅니다.
최초입원일 입원일당이 지급된 
최종입원일 보장재개
퇴원없이
계속 입원 …
보장
(180일)
보장제외
(180일)
보장
(180일) …
ꊴ 제1항의 
--------------------------------------------------------------------------------
[3] page=131 | clause=제3조 | topic=입원
를 계산합니다.
【의료법 제3조(의료기관)】
이 법에서 의료기관이라 함은 의료인이 공중 또는 특수 다수인을 위하여 의
료・조산의 업을 행하는 곳을 말합니다. 의료기관의 종별은 종합병원・병
원・치과병원・한방병원・요양병원・의원・치과의원・한의원 및 조산원으
로 나누어집니다.
--------------------------------------------------------------------------------
[4] page=101 | clause=제3조 | topic=입원
LIG손해보험 101
받은 때에는 최초 

<br>

#### **Task 14-1. 설정값 바꿔가며 실험**

In [27]:
run_experiment(
    overrides={
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "retriever_k": 5,
    },
    questions=["입원 5일 했을 때 보험금 얼마 받을 수 있어?"]
)


[실험 시작]
- chunk_size: 800 -> 1000
- chunk_overlap: 100 -> 200
- retriever_k: 5 -> 5
[PDF 로드 완료] loader=PyPDFLoader, pages=178
[문서 점검]
- 총 페이지 수: 178
- 총 문자 수: 333,378
- 총 토큰 수(gpt-4o-mini 기준): 218,592

[첫 페이지 미리보기]
무배당 
L I G 행 복 한 인 생 보 험(L1 2 . 0 4 ) 
보험안내서 및 약관 
보종 21373 
장기상품팀 2012년 4월 제작 
무배당 LIG행복한인생보험(L12.04)
서울시 강남구 역삼동 649-11 LIG 타워 
고객콜센터 1544-0114 
장기 12-21373-1 
무배당 
L I G 행 복 한 인 생 보 험(L1 2 . 0 4 ) 
보험안내서 및 약관 
보종 21373 
장기상품팀 2012년 4월 제작 
무배당 LIG행복한인생보험(L12.04)
서울시 강남구 역삼동 649-11 LIG 타워 
고객콜센터 15
[노이즈 페이지 제거] kept=149, removed=29
[페이지 메타데이터 보강 완료] 149 pages
[청킹 완료] 390 chunks
[청크 메타데이터 보강 완료] 390 chunks
[청크 미리보기] 상위 3개
chunk_id   : 0
page       : 0
clause_hint: UNKNOWN
topic_hint : 기타
text       : 무배당 
L I G 행 복 한 인 생 보 험(L1 2 . 0 4 ) 
보험안내서 및 약관 
보종 21373 
장기상품팀 2012년 4월 제작 
무배당 LIG행복한인생보험(L12.04)
서울시 강남구 역삼동 649-11 LIG 타워 
고객콜센터 1544-0114 
장기 12-21373-1 
무배당 
L I G 행 복 한 인 생 보 험(L1 2 . 0 4 ) 
chunk_id   : 1
page       : 2
clause_hint: UNKNOWN
topic_hint : 수술
text      

In [28]:
run_experiment(
    overrides={
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "retriever_k": 5,
    },
    questions=["비급여 치료비도 보상이 돼?"]
)


[실험 시작]
- chunk_size: 800 -> 1000
- chunk_overlap: 100 -> 200
- retriever_k: 5 -> 5
[PDF 로드 완료] loader=PyPDFLoader, pages=178
[문서 점검]
- 총 페이지 수: 178
- 총 문자 수: 333,378
- 총 토큰 수(gpt-4o-mini 기준): 218,592

[첫 페이지 미리보기]
무배당 
L I G 행 복 한 인 생 보 험(L1 2 . 0 4 ) 
보험안내서 및 약관 
보종 21373 
장기상품팀 2012년 4월 제작 
무배당 LIG행복한인생보험(L12.04)
서울시 강남구 역삼동 649-11 LIG 타워 
고객콜센터 1544-0114 
장기 12-21373-1 
무배당 
L I G 행 복 한 인 생 보 험(L1 2 . 0 4 ) 
보험안내서 및 약관 
보종 21373 
장기상품팀 2012년 4월 제작 
무배당 LIG행복한인생보험(L12.04)
서울시 강남구 역삼동 649-11 LIG 타워 
고객콜센터 15
[노이즈 페이지 제거] kept=149, removed=29
[페이지 메타데이터 보강 완료] 149 pages
[청킹 완료] 390 chunks
[청크 메타데이터 보강 완료] 390 chunks
[청크 미리보기] 상위 3개
chunk_id   : 0
page       : 0
clause_hint: UNKNOWN
topic_hint : 기타
text       : 무배당 
L I G 행 복 한 인 생 보 험(L1 2 . 0 4 ) 
보험안내서 및 약관 
보종 21373 
장기상품팀 2012년 4월 제작 
무배당 LIG행복한인생보험(L12.04)
서울시 강남구 역삼동 649-11 LIG 타워 
고객콜센터 1544-0114 
장기 12-21373-1 
무배당 
L I G 행 복 한 인 생 보 험(L1 2 . 0 4 ) 
chunk_id   : 1
page       : 2
clause_hint: UNKNOWN
topic_hint : 수술
text      

In [29]:
run_experiment(
    overrides={
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "retriever_k": 5,
    },
    questions=["통원 치료 시 공제금액은 얼마야?"]
)


[실험 시작]
- chunk_size: 800 -> 1000
- chunk_overlap: 100 -> 200
- retriever_k: 5 -> 5
[PDF 로드 완료] loader=PyPDFLoader, pages=178
[문서 점검]
- 총 페이지 수: 178
- 총 문자 수: 333,378
- 총 토큰 수(gpt-4o-mini 기준): 218,592

[첫 페이지 미리보기]
무배당 
L I G 행 복 한 인 생 보 험(L1 2 . 0 4 ) 
보험안내서 및 약관 
보종 21373 
장기상품팀 2012년 4월 제작 
무배당 LIG행복한인생보험(L12.04)
서울시 강남구 역삼동 649-11 LIG 타워 
고객콜센터 1544-0114 
장기 12-21373-1 
무배당 
L I G 행 복 한 인 생 보 험(L1 2 . 0 4 ) 
보험안내서 및 약관 
보종 21373 
장기상품팀 2012년 4월 제작 
무배당 LIG행복한인생보험(L12.04)
서울시 강남구 역삼동 649-11 LIG 타워 
고객콜센터 15
[노이즈 페이지 제거] kept=149, removed=29
[페이지 메타데이터 보강 완료] 149 pages
[청킹 완료] 390 chunks
[청크 메타데이터 보강 완료] 390 chunks
[청크 미리보기] 상위 3개
chunk_id   : 0
page       : 0
clause_hint: UNKNOWN
topic_hint : 기타
text       : 무배당 
L I G 행 복 한 인 생 보 험(L1 2 . 0 4 ) 
보험안내서 및 약관 
보종 21373 
장기상품팀 2012년 4월 제작 
무배당 LIG행복한인생보험(L12.04)
서울시 강남구 역삼동 649-11 LIG 타워 
고객콜센터 1544-0114 
장기 12-21373-1 
무배당 
L I G 행 복 한 인 생 보 험(L1 2 . 0 4 ) 
chunk_id   : 1
page       : 2
clause_hint: UNKNOWN
topic_hint : 수술
text      

<br>

#### **Task 14-2. 추가 실험(영수증 첨부)**

In [88]:
# =========================================================
# Task 17-0. 추가 import
# =========================================================

import re
import json
import base64
import mimetypes
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
import numpy as np

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

In [90]:
# =========================================================
# Task 17-1. 이미지 파일 -> data URL 변환
# =========================================================

def image_file_to_data_url(image_path: str) -> str:
    path = Path(image_path)
    if not path.exists():
        raise FileNotFoundError(f"이미지 파일이 없습니다: {path.resolve()}")

    mime_type, _ = mimetypes.guess_type(str(path))
    if mime_type is None:
        mime_type = "image/png"

    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")

    return f"data:{mime_type};base64,{b64}"

In [91]:
# =========================================================
# Task 17-2. JSON 파싱 헬퍼
# =========================================================

def parse_json_safely(text: str) -> Dict[str, Any]:
    original_text = text
    text = text.strip()

    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1:
        print("[JSON 파싱 실패] JSON 경계를 찾지 못했습니다.")
        print(original_text[:3000])
        raise ValueError("응답에서 JSON 객체를 찾을 수 없습니다.")

    json_str = text[start:end+1]

    try:
        return json.loads(json_str)
    except json.JSONDecodeError as e:
        print("[JSON 파싱 실패]")
        print("에러:", e)
        print(json_str[:4000])
        raise

In [92]:
# =========================================================
# Task 17-3. history_table 전용 이미지 추출 프롬프트
# =========================================================

HISTORY_TABLE_EXTRACTION_PROMPT = """
이미지는 '최근 5년간 보험금 수령 현황' 표입니다.

반드시 JSON 객체 하나만 출력하세요.
설명, 코드블록, 마크다운 금지.

읽어야 할 열:
1. 날짜
2. 재해 원인
3. 재해 유형
4. 진단명
5. 병원 수납액(개인부담금)
6. 보험금 수령액

중요 규칙:
- 병원 수납액 셀 안에 여러 숫자가 줄바꿈으로 들어 있으면 문자열 그대로 모두 보존하세요.
  예: "286140\\n32995\\n18000"
- 보험금 수령액 셀 안에 여러 숫자가 줄바꿈으로 들어 있으면 문자열 그대로 모두 보존하세요.
  예: "222030\\n3700\\n10000"
- 보험금 수령액은 정답 비교용이므로 최대한 정확히 읽으세요.
- 모르면 null.

출력 형식:
{
  "doc_type": "history_table",
  "document_title": "최근 5년간 보험금 수령 현황",
  "rows": [
    {
      "date": "251120",
      "cause": "자전거 타다가 넘어져서 허리와 손 다침",
      "accident_type": "일반 상해",
      "diagnosis": "요추의 염좌 및 긴장",
      "paid_amount": "39300",
      "insurance_paid_actual": "7000"
    }
  ]
}
"""

In [93]:
# =========================================================
# Task 17-4. 이미지 -> 구조화 JSON
# =========================================================

def extract_history_table_from_image(
    image_path: str,
    vision_model: str = "gpt-4o-mini",
    temperature: float = 0.0
) -> Dict[str, Any]:
    data_url = image_file_to_data_url(image_path)

    vision_llm = ChatOpenAI(
        model=vision_model,
        temperature=temperature
    )

    message = HumanMessage(
        content=[
            {"type": "text", "text": HISTORY_TABLE_EXTRACTION_PROMPT},
            {"type": "image_url", "image_url": {"url": data_url}},
        ]
    )

    response = vision_llm.invoke([message])

    print("[비전 모델 원문 응답 일부]")
    print(response.content[:1500])

    parsed = parse_json_safely(response.content)
    print("[이미지 구조화 추출 완료]")
    print("doc_type:", parsed.get("doc_type"))
    return parsed

In [94]:
# =========================================================
# Task 17-5. 복수 금액 셀 분해
# =========================================================

def extract_number_list(value) -> List[int]:
    if value is None:
        return []

    text = str(value)
    nums = re.findall(r"\d[\d,]*", text)
    return [int(n.replace(",", "")) for n in nums]

In [95]:
# =========================================================
# Task 17-6. 구조화 rows -> DataFrame
# - 복수 금액 셀을 여러 행으로 확장
# =========================================================

def history_rows_to_df(extracted: Dict[str, Any]) -> pd.DataFrame:
    rows = extracted.get("rows", [])
    if not rows:
        return pd.DataFrame()

    expanded_rows = []

    for row in rows:
        paid_list = extract_number_list(row.get("paid_amount"))
        actual_list = extract_number_list(row.get("insurance_paid_actual"))

        max_len = max(len(paid_list), len(actual_list), 1)

        if len(paid_list) == 0:
            paid_list = [None] * max_len
        if len(actual_list) == 0:
            actual_list = [None] * max_len

        if len(paid_list) < max_len:
            paid_list += [None] * (max_len - len(paid_list))
        if len(actual_list) < max_len:
            actual_list += [None] * (max_len - len(actual_list))

        for paid, actual in zip(paid_list, actual_list):
            expanded_rows.append({
                "date": row.get("date"),
                "cause": row.get("cause"),
                "accident_type": row.get("accident_type"),
                "diagnosis": row.get("diagnosis"),
                "paid_amount": paid,
                "insurance_paid_actual": actual,  # 정답 비교용
            })

    df = pd.DataFrame(expanded_rows)

    for col in ["paid_amount", "insurance_paid_actual"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

In [96]:
# =========================================================
# Task 17-7. 행별 검색 질의 생성
# - 계산 결과가 아니라 규칙 탐색용 질의
# =========================================================

def build_rule_search_query(row: pd.Series) -> str:
    cause = str(row.get("cause", "") or "")
    accident_type = str(row.get("accident_type", "") or "")
    diagnosis = str(row.get("diagnosis", "") or "")
    paid_amount = row.get("paid_amount", None)

    terms = []

    if accident_type:
        terms.append(accident_type)
    if diagnosis:
        terms.append(diagnosis)
    if cause:
        terms.append(cause)

    # 보험금 계산 규칙 탐색용 키워드 확장
    terms.extend([
        "실손의료비",
        "통원의료비",
        "입원의료비",
        "공제금액",
        "비급여",
        "보상하지 아니하는",
        "자기부담금"
    ])

    # 치과/치아 계열 보강
    diag_no_space = diagnosis.replace(" ", "")
    if "치아" in diag_no_space or "치과" in cause:
        terms.extend(["치과치료", "치아우식", "보상하지 아니하는"])

    # 응급/입원 가능성 보강
    cause_no_space = cause.replace(" ", "")
    if "응급실" in cause_no_space:
        terms.extend(["응급실", "입원의료비", "통원의료비"])

    # 중복 제거
    uniq = []
    for t in terms:
        if t and t not in uniq:
            uniq.append(t)

    return " | ".join(uniq)

In [104]:
# =========================================================
# Task 17-8. 규칙 추출 전용 프롬프트
# - 보험금 숫자 직접 예측 금지
# - 규칙만 추출
# =========================================================

RULE_EXTRACTION_PROMPT = """
당신은 보험 약관 규칙 추출기입니다.
반드시 검색된 약관 내용만 근거로 판단하세요.
보험금 최종 금액을 직접 계산하지 마세요.
설명문 없이 JSON만 출력하세요.

출력 형식:
{{
  "visit_type": "입원" | "통원" | "약국" | "불명확",
  "coverage_possible": true | false | null,
  "deductible_amount": int | null,
  "coverage_ratio": float | null,
  "excluded": true | false | null,
  "exclusion_reason": "문자열 또는 null",
  "rule_basis": [
    "근거 문장 1",
    "근거 문장 2"
  ],
  "needs_manual_review": true | false
}}

판단 규칙:
1. 입력 사례가 입원/통원/약국 중 무엇에 가까운지 약관 근거로 판단하세요.
2. 공제금액이 명확하면 deductible_amount에 숫자로 넣으세요.
   예: 10000, 15000, 20000, 8000
3. 보상비율이 명확하면 coverage_ratio에 숫자로 넣으세요.
   예: 0.9
4. 치과치료, 미용, 건강검진, 보신용 투약 등 명백한 제외면 excluded=true
5. 불명확하면 null 또는 needs_manual_review=true
6. 보험금 최종 금액을 estimated_payout 같은 필드로 출력하지 마세요.

[사례 입력]
{case_info}

[검색된 약관]
{context}
"""

In [105]:
# =========================================================
# Task 17-9. 행별 규칙 추출
# =========================================================

def extract_rule_for_row_with_rag(
    row: pd.Series,
    llm_model: str = "gpt-4o-mini",
    temperature: float = 0.0,
    top_k: int = 5
) -> Dict[str, Any]:
    if _retriever is None:
        raise RuntimeError("initialize(rebuild=True)를 먼저 실행하세요.")

    search_query = build_rule_search_query(row)
    source_docs = _retriever.invoke(search_query)
    source_docs = source_docs[:top_k]

    context = format_docs(source_docs)

    case_info = {
        "date": row.get("date"),
        "cause": row.get("cause"),
        "accident_type": row.get("accident_type"),
        "diagnosis": row.get("diagnosis"),
        "paid_amount": int(row["paid_amount"]) if pd.notnull(row.get("paid_amount")) else None
    }

    prompt = RULE_EXTRACTION_PROMPT.format(
        case_info=json.dumps(case_info, ensure_ascii=False, indent=2),
        context=context
    )

    llm = ChatOpenAI(
        model=llm_model,
        temperature=temperature
    )

    message = HumanMessage(content=[{"type": "text", "text": prompt}])
    response = llm.invoke([message])

    rule_json = parse_json_safely(response.content)

    return {
        "search_query": search_query,
        "source_docs": source_docs,
        "rule_json": rule_json
    }

In [106]:
# =========================================================
# Task 17-10. Python 계산 함수
# - LLM이 준 규칙을 바탕으로만 계산
# =========================================================

def calculate_payout_by_rule(paid_amount: Optional[float], rule_json: Dict[str, Any]) -> Optional[int]:
    if paid_amount is None or pd.isnull(paid_amount):
        return None

    paid_amount = float(paid_amount)

    coverage_possible = rule_json.get("coverage_possible")
    excluded = rule_json.get("excluded")
    deductible_amount = rule_json.get("deductible_amount")
    coverage_ratio = rule_json.get("coverage_ratio")

    # 명시적 제외
    if excluded is True:
        return 0

    # 보상 불가
    if coverage_possible is False:
        return 0

    # 비율 있으면 적용
    if coverage_ratio is not None:
        try:
            ratio = float(coverage_ratio)
        except:
            ratio = None
    else:
        ratio = None

    # 공제금액 있으면 적용
    if deductible_amount is not None:
        try:
            deductible = int(deductible_amount)
        except:
            deductible = None
    else:
        deductible = None

    amount = paid_amount

    if deductible is not None:
        amount = amount - deductible

    if ratio is not None:
        amount = amount * ratio

    amount = max(amount, 0)
    return int(round(amount))

In [107]:
# =========================================================
# Task 17-11. 전체 행에 규칙 추출 + Python 계산 적용
# =========================================================

def apply_rule_based_rag_prediction(
    df: pd.DataFrame,
    llm_model: str = "gpt-4o-mini",
    temperature: float = 0.0,
    top_k: int = 5
) -> pd.DataFrame:
    if df.empty:
        return df.copy()

    result_rows = []

    for idx, row in df.iterrows():
        rag_result = extract_rule_for_row_with_rag(
            row=row,
            llm_model=llm_model,
            temperature=temperature,
            top_k=top_k
        )

        rule_json = rag_result["rule_json"]
        predicted_payout = calculate_payout_by_rule(row.get("paid_amount"), rule_json)

        result_row = row.to_dict()
        result_row["predicted_payout"] = predicted_payout
        result_row["visit_type_pred"] = rule_json.get("visit_type")
        result_row["coverage_possible_pred"] = rule_json.get("coverage_possible")
        result_row["deductible_amount_pred"] = rule_json.get("deductible_amount")
        result_row["coverage_ratio_pred"] = rule_json.get("coverage_ratio")
        result_row["excluded_pred"] = rule_json.get("excluded")
        result_row["needs_manual_review_pred"] = rule_json.get("needs_manual_review")
        result_row["rule_basis"] = " | ".join(rule_json.get("rule_basis", []))
        result_row["search_query"] = rag_result["search_query"]

        result_rows.append(result_row)

        print(
            f"[{idx+1}/{len(df)}] "
            f"date={row.get('date')} | "
            f"paid={row.get('paid_amount')} | "
            f"deduct={rule_json.get('deductible_amount')} | "
            f"ratio={rule_json.get('coverage_ratio')} | "
            f"excluded={rule_json.get('excluded')} | "
            f"predicted={predicted_payout}"
        )

    return pd.DataFrame(result_rows)

In [108]:
# =========================================================
# Task 17-12. 예측 결과 평가
# - 실제값이 있는 행만 따로 평가
# - 전체 예측합도 함께 출력
# =========================================================

def evaluate_prediction_df(result_df: pd.DataFrame) -> Dict[str, Any]:
    if result_df.empty:
        return {}

    actual_mask = result_df["insurance_paid_actual"].notna()

    actual_total_all = result_df["insurance_paid_actual"].fillna(0).sum()
    predicted_total_all = result_df["predicted_payout"].fillna(0).sum()

    actual_total_eval = result_df.loc[actual_mask, "insurance_paid_actual"].fillna(0).sum()
    predicted_total_eval = result_df.loc[actual_mask, "predicted_payout"].fillna(0).sum()

    abs_error_eval = abs(actual_total_eval - predicted_total_eval)
    error_rate_eval = abs_error_eval / actual_total_eval if actual_total_eval > 0 else None

    summary = {
        "actual_total_all_detected": int(actual_total_all),
        "predicted_total_all": int(predicted_total_all),
        "actual_total_eval_rows": int(actual_total_eval),
        "predicted_total_eval_rows": int(predicted_total_eval),
        "absolute_error_eval_rows": int(abs_error_eval),
        "error_rate_eval_rows": error_rate_eval,
        "actual_rows_count": int(actual_mask.sum()),
        "all_rows_count": int(len(result_df))
    }

    print("=" * 90)
    print("[예측 결과 평가]")
    print("=" * 90)
    print(f"실제값이 읽힌 행 수        : {summary['actual_rows_count']} / {summary['all_rows_count']}")
    print(f"실제 총 보험금(읽힌 행 기준): {summary['actual_total_eval_rows']:,}원")
    print(f"예측 총 보험금(동일 행 기준): {summary['predicted_total_eval_rows']:,}원")
    print(f"절대 오차                  : {summary['absolute_error_eval_rows']:,}원")
    if summary["error_rate_eval_rows"] is not None:
        print(f"오차율                     : {summary['error_rate_eval_rows']:.4%}")
    else:
        print("오차율                     : 계산 불가")

    print("-" * 90)
    print(f"예측 총 보험금(전체 행)     : {summary['predicted_total_all']:,}원")
    print(f"실제 총 보험금(감지된 값 합): {summary['actual_total_all_detected']:,}원")

    return summary

In [109]:
# =========================================================
# Task 17-13. 전체 통합 실행
# - 이미지 -> 표 추출 -> DataFrame -> 행별 규칙 추출 -> Python 계산 -> 평가
# =========================================================

def run_history_table_rag_pipeline(
    image_path: str,
    vision_model: str = "gpt-4o-mini",
    payout_model: str = "gpt-4o-mini",
    temperature: float = 0.0,
    top_k: int = 5,
    verbose: bool = True
):
    extracted = extract_history_table_from_image(
        image_path=image_path,
        vision_model=vision_model,
        temperature=0.0
    )

    df = history_rows_to_df(extracted)

    if verbose:
        print("\n[추출된 DataFrame]")
        display(df)

    result_df = apply_rule_based_rag_prediction(
        df=df,
        llm_model=payout_model,
        temperature=temperature,
        top_k=top_k
    )

    if verbose:
        print("\n[예측 결과 DataFrame]")
        display(result_df[[
            "date", "cause", "accident_type", "diagnosis",
            "paid_amount", "insurance_paid_actual",
            "predicted_payout", "visit_type_pred",
            "deductible_amount_pred", "coverage_ratio_pred",
            "excluded_pred", "needs_manual_review_pred"
        ]])

    evaluation = evaluate_prediction_df(result_df)

    return {
        "extracted": extracted,
        "df": df,
        "result_df": result_df,
        "evaluation": evaluation
    }

In [110]:
# =========================================================
# Task 17-14. 최종 실행
# - 반드시 그 전에 initialize(rebuild=True) 실행
# =========================================================

image_path = "/content/drive/MyDrive/DS_8th/09_LLM 활용/data/test_영수증.png"

pipeline_result = run_history_table_rag_pipeline(
    image_path=image_path,
    vision_model="gpt-4o-mini",
    payout_model="gpt-4o-mini",
    temperature=0.0,
    top_k=5,
    verbose=True
)

[비전 모델 원문 응답 일부]
{
  "doc_type": "history_table",
  "document_title": "최근 5년간 보험금 수령 현황",
  "rows": [
    {
      "date": "251120",
      "cause": "자전거 타다가 넘어져서 허리와 손 다침",
      "accident_type": "일반 상해",
      "diagnosis": "요추의 염좌 및 긴장",
      "paid_amount": "39300",
      "insurance_paid_actual": "7000"
    },
    {
      "date": "251130",
      "cause": "무리한 동작",
      "accident_type": "일반 상해",
      "diagnosis": "상세 불명의 표재성 손상",
      "paid_amount": "15600",
      "insurance_paid_actual": "null"
    },
    {
      "date": "250627",
      "cause": "낚시 중 낚시바늘에 찔려 소독",
      "accident_type": "일반 상해",
      "diagnosis": "상세불명의 표재성 손상",
      "paid_amount": "13400",
      "insurance_paid_actual": "null"
    },
    {
      "date": "250616",
      "cause": "무리한 동작",
      "accident_type": "질병",
      "diagnosis": "요통",
      "paid_amount": "32906",
      "insurance_paid_actual": "null"
    },
    {
      "date": "240826",
      "cause": "코로나19 바이러스",
      "accident_type": "질병",
      "dia

,date,cause,accident_type,diagnosis,paid_amount,insurance_paid_actual
0,251120,자전거 타다가 넘어져서 허리와 손 다침,일반 상해,요추의 염좌 및 긴장,39300,7000.0
1,251130,무리한 동작,일반 상해,상세 불명의 표재성 손상,15600,NaN
2,250627,낚시 중 낚시바늘에 찔려 소독,일반 상해,상세불명의 표재성 손상,13400,NaN
3,250616,무리한 동작,질병,요통,32906,NaN
4,240826,코로나19 바이러스,질병,상세불명의 열,25200,NaN
5,240612,안구 통증,질병,눈통증,19812,NaN
6,221226,스쿠터골프 치던 중 무리하게 뒤틀어쳐 갈비뼈 통증,일반상해,상세불명의 흉통,33956,NaN
7,220705,치과 진료,질병,치아우식,16651,NaN
8,211104,치과 진료,질병,치아 우식,25603,NaN
9,211205,치과 진료,질병,치아 우식,24964,NaN


[1/14] date=251120 | paid=39300 | deduct=None | ratio=None | excluded=False | predicted=39300
[2/14] date=251130 | paid=15600 | deduct=None | ratio=None | excluded=False | predicted=15600
[3/14] date=250627 | paid=13400 | deduct=None | ratio=None | excluded=False | predicted=13400
[4/14] date=250616 | paid=32906 | deduct=None | ratio=None | excluded=False | predicted=32906
[5/14] date=240826 | paid=25200 | deduct=None | ratio=None | excluded=None | predicted=25200
[6/14] date=240612 | paid=19812 | deduct=None | ratio=None | excluded=True | predicted=0
[7/14] date=221226 | paid=33956 | deduct=None | ratio=None | excluded=True | predicted=0
[8/14] date=220705 | paid=16651 | deduct=None | ratio=None | excluded=True | predicted=0
[9/14] date=211104 | paid=25603 | deduct=None | ratio=None | excluded=True | predicted=0
[10/14] date=211205 | paid=24964 | deduct=None | ratio=None | excluded=True | predicted=0
[11/14] date=210310 | paid=21452 | deduct=None | ratio=0.9 | excluded=False | predict

,date,cause,accident_type,diagnosis,paid_amount,insurance_paid_actual,predicted_payout,visit_type_pred,deductible_amount_pred,coverage_ratio_pred,excluded_pred,needs_manual_review_pred
0,251120,자전거 타다가 넘어져서 허리와 손 다침,일반 상해,요추의 염좌 및 긴장,39300,7000.0,39300,입원,None,NaN,False,False
1,251130,무리한 동작,일반 상해,상세 불명의 표재성 손상,15600,NaN,15600,불명확,None,NaN,False,True
2,250627,낚시 중 낚시바늘에 찔려 소독,일반 상해,상세불명의 표재성 손상,13400,NaN,13400,통원,None,NaN,False,False
3,250616,무리한 동작,질병,요통,32906,NaN,32906,입원,None,NaN,False,False
4,240826,코로나19 바이러스,질병,상세불명의 열,25200,NaN,25200,불명확,None,NaN,None,True
5,240612,안구 통증,질병,눈통증,19812,NaN,0,통원,None,NaN,True,False
6,221226,스쿠터골프 치던 중 무리하게 뒤틀어쳐 갈비뼈 통증,일반상해,상세불명의 흉통,33956,NaN,0,통원,None,NaN,True,False
7,220705,치과 진료,질병,치아우식,16651,NaN,0,불명확,None,NaN,True,False
8,211104,치과 진료,질병,치아 우식,25603,NaN,0,불명확,None,NaN,True,False
9,211205,치과 진료,질병,치아 우식,24964,NaN,0,불명확,None,NaN,True,False


[예측 결과 평가]
실제값이 읽힌 행 수        : 4 / 14
실제 총 보험금(읽힌 행 기준): 242,730원
예측 총 보험금(동일 행 기준): 376,435원
절대 오차                  : 133,705원
오차율                     : 55.0838%
------------------------------------------------------------------------------------------
예측 총 보험금(전체 행)     : 482,848원
실제 총 보험금(감지된 값 합): 242,730원
